# 02 · Neural course baselines trained from scratch: CTC, hybrid CTC/attention, RNN-T

Runs on Kaggle with a GPU. **Settings → Accelerator: GPU, Internet: On.** No manual uploads.
Set `SYSTEM` to `ctc_scratch`, `aed_hybrid_scratch` or `rnnt_scratch` (one Kaggle session each).

`scripts/train_scratch.py` trains on the train split, chooses the checkpoint and the decoding
settings on dev, and decodes test once at the end (settings: `configs/train_scratch.yaml`).
`TRIAL = True` runs one epoch on 200 train clips and 20 dev/test clips into `/kaggle/working/trial`.

Output: `/kaggle/working/results_<SYSTEM>.zip` (predictions, `.meta.json`, training log) and the
checkpoint `/kaggle/working/<SYSTEM>_best.pt` (not in the repository).

In [ ]:
REPO_URL = "https://github.com/eneserman/NLP.git"  # the public repository
BRANCH = "main"
SYSTEM = "ctc_scratch"
TRIAL = False

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -q -b $BRANCH $REPO_URL repo
%cd /kaggle/working/repo
!git log -1 --format='commit %H'
!pip install -q -r requirements/train.txt
!pip freeze | grep -iE '^(torch|numpy|pandas|pyarrow|jiwer|pyyaml)=='

In [ ]:
!python -m pytest -q tests/test_scratch.py tests/test_gmm_hmm.py tests/test_inference.py

In [ ]:
!python scripts/unpack_audio.py

In [ ]:
import subprocess, sys
extra = ["--epochs", "1", "--limit-train", "200", "--limit-eval", "20", "--out-dir", "/kaggle/working/trial"] if TRIAL else []
r = subprocess.run([sys.executable, "scripts/train_scratch.py", "--system", SYSTEM, *extra])
print("exit code", r.returncode)

In [ ]:
if TRIAL:
    !cat /kaggle/working/trial/train_logs/$SYSTEM.json | head -c 3000
    !head -n 3 /kaggle/working/trial/predictions/$SYSTEM.dev.jsonl
else:
    !cp work/models/$SYSTEM/best.pt /kaggle/working/{SYSTEM}_best.pt
    !zip -q -r /kaggle/working/results_{SYSTEM}.zip results/predictions/{SYSTEM}.* results/train_logs/{SYSTEM}.json
    !ls -lh /kaggle/working